# 06 · PostGIS: store all roofs, spatial index, partitioning, fast queries

**Run this in Google Colab.** It takes about 25–40 minutes.

What it does *(Module 2: Postgres-based spatial storage, real-time queries, partitioning)*:
1. Installs **PostgreSQL + PostGIS** inside Colab.
2. Loads all ~13 million roofs (centre point + all features) and the 14 districts.
3. **Speed test 1 — spatial index:** "roofs within 1 km of Kochi" **without** and **with** a GiST index.
4. **Speed test 2 — partitioning:** one district's total on a normal table vs a table **partitioned by district**.
5. **Decision-support queries:** nearest roofs to a point, best roofs near Thiruvananthapuram, a PostGIS spatial join.

The database lives only in this Colab session; re-run this notebook to rebuild it.
Everything printed is also saved to `MyDrive/rooftop_solar/logs/06_log.txt`.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the inputs to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
RAW = f'{PROJECT}/data/raw'
FEATURES = f'{PROJECT}/data/features'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'
for folder in (RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/06_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


T_START = time.time()
shutil.copytree(f'{FEATURES}/roof_features_v2', f'{LOCAL}/roof_features_v2', dirs_exist_ok=True)
shutil.copy(f'{RAW}/kerala_boundaries.parquet', f'{LOCAL}/kerala_boundaries.parquet')
log(f'Inputs copied to {LOCAL} in {time.time() - T_START:.0f} s')

In [ ]:
# Step 2: install PostgreSQL + PostGIS, start the server, create the database "solar"
!apt-get -qq update > /dev/null && apt-get -qq install -y postgresql postgis > /dev/null
!service postgresql start
!sudo -u postgres psql -q -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -q -c "CREATE DATABASE solar;" || true
!sudo -u postgres psql -q -d solar -c "CREATE EXTENSION IF NOT EXISTS postgis;"
!pip install -q duckdb==2.0.0 psycopg2-binary

import pandas as pd
import psycopg2

pg = psycopg2.connect(host='localhost', dbname='solar', user='postgres', password='postgres')
pg.autocommit = True
cur = pg.cursor()


def sql(query):
    """Run one SQL statement in PostGIS; return the rows if it gives any."""
    cur.execute(query)
    return cur.fetchall() if cur.description else None


def sql_df(query):
    """Run a query and return the result as a pandas table."""
    cur.execute(query)
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


TIMES = []


def timed(label, query, runs=2):
    """Time a query with EXPLAIN ANALYZE (keeps the last run, i.e. warm cache) and note the tables it read."""
    for _ in range(runs):
        cur.execute('EXPLAIN (ANALYZE, FORMAT JSON) ' + query)
        plan = cur.fetchone()[0][0]
    tables = set()

    def walk(node):
        if 'Relation Name' in node:
            tables.add(node['Relation Name'])
        for child in node.get('Plans', []):
            walk(child)

    walk(plan['Plan'])
    ms = plan['Execution Time']
    TIMES.append({'test': label, 'ms': round(ms, 1), 'tables_read': len(tables)})
    log(f'  {label}: {ms:,.1f} ms  ({len(tables)} table(s) read)')
    return ms


log(sql('SELECT postgis_full_version()')[0][0][:60])

In [ ]:
# Step 3: load all roofs and the districts into PostGIS (DuckDB copies them straight into PostgreSQL)
import duckdb

t = time.time()
sql('DROP TABLE IF EXISTS roofs_part, roofs, districts CASCADE')    # start clean if this is a re-run
con = duckdb.connect()
con.sql("INSTALL spatial; LOAD spatial; INSTALL postgres; LOAD postgres; SET geometry_always_xy = true;")
con.sql("ATTACH 'host=localhost dbname=solar user=postgres password=postgres' AS pg (TYPE postgres)")
con.execute(f"""
    CREATE TABLE pg.public.roofs AS
    SELECT id, district, source, confidence, low_confidence, area_m2, perimeter_m, compactness,
           tree_pct, tree_pct_140m, tree_pct_300m, roof_seen_as, pvout_kwh_per_kwp, h3_9, bldgs_in_hex,
           usable_m2, kwp, kwh_year, centre_lon, centre_lat
    FROM read_parquet('{LOCAL}/roof_features_v2/*/*.parquet', hive_partitioning = true)
""").fetchall()
con.execute(f"""
    CREATE TABLE pg.public.districts AS
    SELECT name AS district, ST_AsWKB(geometry) AS wkb
    FROM read_parquet('{LOCAL}/kerala_boundaries.parquet') WHERE subtype = 'county'
""").fetchall()
con.close()
log(f'Copied into PostgreSQL in {(time.time() - t) / 60:.1f} min')

# Real PostGIS geometry columns, filled automatically from the stored values
t = time.time()
sql("""ALTER TABLE roofs ADD COLUMN geom geometry(Point, 4326)
       GENERATED ALWAYS AS (ST_SetSRID(ST_MakePoint(centre_lon, centre_lat), 4326)) STORED""")
sql("""ALTER TABLE districts ADD COLUMN geom geometry(MultiPolygon, 4326)
       GENERATED ALWAYS AS (ST_Multi(ST_SetSRID(ST_GeomFromWKB(wkb), 4326))) STORED""")
sql('ANALYZE roofs')
sql('ANALYZE districts')
ROOFS = sql('SELECT count(*) FROM roofs')[0][0]
N_DISTRICTS = sql('SELECT count(*) FROM districts')[0][0]
SIZE = sql("SELECT pg_size_pretty(pg_total_relation_size('roofs'))")[0][0]
log(f'Geometry columns added in {(time.time() - t) / 60:.1f} min | roofs: {ROOFS:,} | '
    f'districts: {N_DISTRICTS} | roofs table size: {SIZE}')

In [ ]:
# Step 4: speed test 1 - "how many roofs, and how much solar, within 1 km of Kochi?" without and with a spatial index
KOCHI = "ST_SetSRID(ST_MakePoint(76.2673, 9.9312), 4326)"
NEAR_KOCHI = f"""
    SELECT count(*) AS roofs, round(sum(kwh_year)::numeric / 1e6, 2) AS gwh_per_year
    FROM roofs WHERE ST_DWithin(geom::geography, {KOCHI}::geography, 1000)
"""
log('\nSPEED TEST 1: spatial index (GiST)')
timed('roofs within 1 km of Kochi - no index', NEAR_KOCHI)

t = time.time()
sql("SET maintenance_work_mem = '2GB'")
sql('CREATE INDEX roofs_geog_idx ON roofs USING gist ((geom::geography))')   # for distances in metres
sql('CREATE INDEX roofs_geom_idx ON roofs USING gist (geom)')                # for joins and nearest-roof
sql('CREATE INDEX districts_geom_idx ON districts USING gist (geom)')
sql('ANALYZE roofs')
log(f'  (building the spatial indexes took {(time.time() - t) / 60:.1f} min)')

timed('roofs within 1 km of Kochi - with GiST index', NEAR_KOCHI)
log_table(sql_df(NEAR_KOCHI))

In [ ]:
# Step 5: speed test 2 - one district's total on a normal table vs a table partitioned by district
t = time.time()
districts = [r[0] for r in sql('SELECT DISTINCT district FROM roofs ORDER BY 1')]
sql('DROP TABLE IF EXISTS roofs_part')
sql('CREATE TABLE roofs_part (LIKE roofs) PARTITION BY LIST (district)')
for i, d in enumerate(districts, 1):
    sql(f"CREATE TABLE roofs_part_{i:02d} PARTITION OF roofs_part FOR VALUES IN ('{d}')")
sql('INSERT INTO roofs_part SELECT * FROM roofs')
sql('ANALYZE roofs_part')
log(f'\nPartitioned copy (one partition per district, {len(districts)} partitions) made in '
    f'{(time.time() - t) / 60:.1f} min')

WAYANAD = "SELECT count(*), round(sum(kwh_year)::numeric / 1e9, 3) AS twh_per_year FROM {} WHERE district = 'Wayanad'"
log('SPEED TEST 2: partitioning')
timed('Wayanad total - normal table', WAYANAD.format('roofs'))
timed('Wayanad total - partitioned table', WAYANAD.format('roofs_part'))
log_table(sql_df(WAYANAD.format('roofs_part')))

In [ ]:
# Step 6: decision-support queries (what PostGIS is good at)
log('\nQUERY A: PostGIS spatial join - roofs inside the Wayanad polygon (should match the district column)')
timed('spatial join: roofs inside Wayanad polygon', """
    SELECT count(*) FROM districts d JOIN roofs r ON ST_Contains(d.geom, r.geom) WHERE d.district = 'Wayanad'
""", runs=1)
log_table(sql_df("""
    SELECT (SELECT count(*) FROM districts d JOIN roofs r ON ST_Contains(d.geom, r.geom)
            WHERE d.district = 'Wayanad') AS by_polygon,
           (SELECT count(*) FROM roofs WHERE district = 'Wayanad') AS by_column
"""))

TVM = "ST_SetSRID(ST_MakePoint(76.9366, 8.5241), 4326)"
log('\nQUERY B: the 5 roofs nearest to Thiruvananthapuram centre (index-based nearest neighbour)')
NEAREST = f"""
    SELECT id, round(area_m2::numeric, 1) AS area_m2, round(kwp::numeric, 1) AS kwp,
           round(ST_Distance(geom::geography, {TVM}::geography)::numeric, 1) AS metres_away
    FROM roofs ORDER BY geom <-> {TVM} LIMIT 5
"""
timed('5 nearest roofs to a point', NEAREST)
log_table(sql_df(NEAREST))

log('\nQUERY C: best 10 roofs within 2 km of Thiruvananthapuram centre (seen as built-up, little tree cover)')
BEST = f"""
    SELECT id, round(area_m2::numeric) AS area_m2, round(kwp::numeric, 1) AS kwp,
           round(kwh_year::numeric) AS units_per_year, tree_pct_140m
    FROM roofs
    WHERE ST_DWithin(geom::geography, {TVM}::geography, 2000)
      AND roof_seen_as = 'built-up' AND tree_pct_140m < 30
    ORDER BY kwh_year DESC LIMIT 10
"""
timed('best roofs within 2 km', BEST)
log_table(sql_df(BEST))

In [ ]:
# Step 7: summary of all timings (send this to Claude)
log('\nALL QUERY TIMES')
log_table(pd.DataFrame(TIMES), 'postgis_query_times.csv')
log(f'Total time {(time.time() - T_START) / 60:.1f} min')
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/06_log.txt`.